# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AshenDary/Week1_RunTheStarterNotebooks/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

I am framing Lane 2 as a Scoring / Ranking task. The model predicts the probability that a page needs attention (a score between 0 and 1). We use this score to sort all pages in a ranked priority queue from highest to lowest review priority. This supports the editor's decision on which pages to review first when time is limited.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

My target is a binary proxy flag called `needs_review`:
- `1` if `trend_direction == 'down'` AND `impressions_90d >= 100`
- `0` otherwise

In our dataset, this marks 13,152 of 30,000 pages (43.84%) as `needs_review = 1`.

This is a proxy because we do not have a direct signal that guarantees a refresh will succeed. Falling traffic on pages with decent search views is a practical observable signal for pages that an editor should inspect first. 

Note: `trend_direction` will only be used to define this target and will not be used as a model feature.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

My primary success metric is Precision@K (such as Precision@10 or Precision@20). 

Precision@K measures the proportion of recommended pages in the top K positions that actually need a review (`needs_review = 1`). 

This metric directly supports our content action because an editor will only review a small batch of pages at a time. High precision at the top of the list ensures we do not waste the editor's time with bad recommendations. As a secondary metric, I will use ROC-AUC to evaluate the overall ranking ability across the entire dataset.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

The unit of analysis is one unique content page (`content_id`) for a specific client. Each row represents performance metrics for a single page over observed time periods.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

DATA_FILENAME = "content_refresh_anonymized.csv"
RELATIVE_DATA_PATH = Path("data") / "raw" / DATA_FILENAME


def find_data_file() -> Path:
    """Find the starter CSV whether the notebook runs from repo root or work/notebooks."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / RELATIVE_DATA_PATH
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {RELATIVE_DATA_PATH} from {Path.cwd()}")


data_path = find_data_file()
df = pd.read_csv(data_path)

required_columns = {
    "content_id",
    "client_id",
    "trend_direction",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "content_age_days",
    "ctr",
}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing expected columns: {sorted(missing_columns)}")

df["target_needs_review"] = (
    df["trend_direction"].eq("down") & df["impressions_90d"].ge(100)
).astype(int)

# trend_direction is shown here only to audit the target logic; do not use it as a model feature.
preview_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "content_age_days",
    "ctr",
    "target_needs_review",
]

display(df[preview_columns].head())

positive_count = int(df["target_needs_review"].sum())
duplicate_content_ids = int(df.duplicated(subset=["content_id"]).sum())
print(f"Dataframe shape: {df.shape}")
print(f"Rows where target_needs_review == 1: {positive_count}")
print(f"Duplicate content_id rows: {duplicate_content_ids}")


## 5. Why ML beats a fixed rule here

A fixed rule creates hard cutoffs that treat all flagged pages equally. It cannot distinguish between a high-impression page at position 2 with a low CTR versus an old page at position 18 that has not been updated recently. Additionally, a simple rule flags a massive bucket of 13,152 pages without ranking which ones need attention first.

ML beats a fixed rule because it evaluates multiple continuous signals simultaneously—including `content_age_days`, `days_since_last_update`, `avg_position`, `ctr`, and `impressions_90d`. Instead of a rigid yes/no label, ML provides a smooth probability score (0 to 1) that ranks pages into a prioritized review queue for the editor. 

Note: While `trend_direction` defines our target, it will be excluded from the model's feature set to prevent data leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before submitting, I confirm that my framing satisfies all project requirements:
- Task Type: Named as a Scoring / Ranking task outputting a probability score (0 to 1).
- Target / Proxy: Defined as `needs_review` using only allowed raw columns (`trend_direction == 'down'` AND `impressions_90d >= 100`).
- Success Metric: Selected Precision@K (e.g., Precision@10) to optimize the top of the review queue for the editor.
- Unit of Analysis: Verified in code that 1 row = 1 unique content page (`content_id`).
- Why ML Beats Rules: Explained using allowed continuous features while guarding against data leakage (`trend_direction`).
- Real Content Action: Connects the model's output score to a prioritized queue that helps a human editor decide which pages to inspect first.
- Honest Claims: Used cautious wording and avoided claiming that a refresh guarantees traffic recovery.